[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-09-geographic-viz.ipynb#scrollTo=aa191001)

---
# Day 9 · Geographic Visualization
**certified-journeys / altair-certified** · Day 9 · Maps, Choropleths & Projections

> **Goal for today:** Render world maps from TopoJSON, join data to geographic features with `transform_lookup`, build a choropleth, overlay city point markers, and control map projections.

In [ ]:
%pip install -q altair vega-datasets

## How Altair Geographic Visualization Works

Altair renders maps through the **Vega-Lite geoshape mark**:

```
TopoJSON topology
  ↓  (feature extraction via url + feature name)
GeoJSON features (one per country / state / etc.)
  ↓  transform_lookup: join your data ON feature id/name
Choropleth: features colored by your joined field
  ↓  layer: add point marks, text labels, etc.
Final layered map
```

Key insight: **the TopoJSON itself has only geometry — no population, no GDP**. Your data lives in a separate DataFrame. `transform_lookup` is the bridge.

## Step 1 · Base World Map with mark_geoshape

`mark_geoshape()` renders geographic features (polygons, lines, points) from a TopoJSON or GeoJSON source. The source is loaded via `alt.topo_feature(url, feature_name)` — this extracts the named feature collection from the topology.

In [ ]:
import altair as alt
from vega_datasets import data

# URL to the world countries TopoJSON bundled with vega-datasets
world_topo_url = data.world_110m.url  # 110m resolution TopoJSON

# Extract the 'countries' feature collection from the topology
countries_source = alt.topo_feature(world_topo_url, feature="countries")

base_map = (
    alt.Chart(countries_source)
    .mark_geoshape(
        fill="#d0e8f0",    # country fill color
        stroke="white",    # border between countries
        strokeWidth=0.5,
    )
    .project("naturalEarth1")  # map projection
    .properties(
        title="World Base Map (Natural Earth projection)",
        width=700,
        height=400
    )
)

base_map

**What just happened?**

- `data.world_110m.url` gives us the CDN URL for the 110m-resolution world TopoJSON.
- `alt.topo_feature(url, feature='countries')` tells Vega-Lite which feature collection inside the topology to render.
- `mark_geoshape()` renders each feature as a filled polygon.
- `.project('naturalEarth1')` applies the Natural Earth projection — one of dozens available in Vega-Lite.

## Step 2 · Join Data to Map Features with transform_lookup

`transform_lookup` is the geographic equivalent of a SQL `LEFT JOIN`. It joins your data DataFrame to the GeoJSON features using a shared key field.

```
chart.transform_lookup(
    lookup='id',           # field in the GeoJSON features
    from_=alt.LookupData(source, 'id', ['field1', 'field2'])
                           # ↑ your DataFrame    ↑ join key   ↑ fields to bring in
)
```

The world_110m countries have numeric ISO-3166 country IDs in their `id` property. We need to join a DataFrame that has the same numeric country IDs.

In [ ]:
import pandas as pd

# Load population data — vega-datasets provides this
pop_data = data.population_engineers_hurricanes()
# This has state-level US data — not ideal for a world map.
# Instead let's load the gapminder data which has country info
gap = data.gapminder()

# The world_110m TopoJSON uses numeric country IDs.
# vega-datasets also provides a lookup table: country ISO codes
# We'll use the 'country-ids' approach manually for clarity.
# A clean approach: use the gapminder_all_years dataset + the 'country_id' numeric code.
# For this notebook we'll use a simpler dataset included in vega-datasets:
# 'lookup_data' approach with the countries dataset

# Load the gapminder dataset and filter to one year
gap_2000 = gap[gap["year"] == 2000].copy()
print("Gapminder columns:", gap_2000.columns.tolist())
print("Sample rows:")
print(gap_2000[["country", "cluster", "pop", "life_expect", "fertility"]].head(5))

## Step 3 · Choropleth Map — Color by a Numeric Field

A choropleth colors geographic regions by a data value. The pattern:
1. Start from the topology source (geoshape)
2. `transform_lookup` to join your data
3. Encode the `color` channel with the joined field

For the world map, we'll use a pre-built Altair example approach with the `unemployment` dataset mapped to US states (which has matching FIPS codes), then show the world approach with `gapminder` life expectancy joined via country id.

In [ ]:
# US State Choropleth — FIPS codes match directly
us_states_url = data.us_10m.url   # US TopoJSON
unemployment = data.unemployment()

# The TopoJSON 'states' feature has numeric FIPS 'id' that matches unemployment's 'id'
us_choropleth = (
    alt.Chart(
        alt.topo_feature(us_states_url, "states")
    )
    .mark_geoshape()
    .encode(
        color=alt.Color(
            "rate:Q",
            title="Unemployment %",
            scale=alt.Scale(scheme="orangered"),
        ),
        tooltip=["id:O", "rate:Q"],
    )
    .transform_lookup(
        lookup="id",                               # field in the TopoJSON features
        from_=alt.LookupData(
            unemployment,   # DataFrame to join
            "id",           # matching key in the DataFrame
            ["rate"]        # fields to bring into the chart
        )
    )
    .project("albersUsa")    # US-centric Albers equal-area projection
    .properties(
        title="US Unemployment Rate by State",
        width=650,
        height=380
    )
)

us_choropleth

**What just happened?**

- `transform_lookup(lookup='id', from_=alt.LookupData(df, 'id', ['rate']))` joins the unemployment DataFrame to the TopoJSON features on the numeric FIPS `id`.
- After the join, `rate` is available as an encoding field — `color='rate:Q'` colors each state.
- `scale=alt.Scale(scheme='orangered')` uses a sequential color scheme (light orange → dark red).
- States with no matching data appear as the default `mark_geoshape` fill color (usually grey).

## Step 4 · Add Country-Name Labels with mark_text

Labels on geographic features require a second chart layer using `mark_text`. The text marks need `longitude` and `latitude` coordinates — computed from the feature centroids.

We use `transform_calculate` or a lookup to add centroid coordinates, then layer text on the map.

In [ ]:
# Build a US state map with abbreviated state labels
# We need state name + centroid coordinates — the us-state-capitals dataset has this
capitals = data.us_state_capitals()
print("Capitals columns:", capitals.columns.tolist())

# Base choropleth (same as before)
state_base = (
    alt.Chart(alt.topo_feature(us_states_url, "states"))
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        color=alt.Color("rate:Q", scale=alt.Scale(scheme="blues"), title="Unemployment %"),
    )
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(unemployment, "id", ["rate"])
    )
    .project("albersUsa")
)

# Text labels layer — capital city points as state markers
state_labels = (
    alt.Chart(capitals)
    .mark_text(fontSize=7, color="#333", fontWeight="bold")
    .encode(
        longitude="lon:Q",   # Vega-Lite geographic longitude encoding
        latitude="lat:Q",    # Vega-Lite geographic latitude encoding
        text="state:N",      # label text
    )
    .project("albersUsa")
)

# Layer map + labels
labeled_map = (
    alt.layer(state_base, state_labels)
    .properties(title="US Unemployment with State Labels", width=650, height=380)
)

labeled_map

**What just happened?**

- `longitude` and `latitude` are **special geographic encoding channels** — Vega-Lite understands these and applies the map projection automatically.
- `alt.layer(base, labels)` stacks the label chart on top of the choropleth.
- **Both layers must use the same `project()` call** — otherwise coordinates won't align.
- For small states, labels will overlap — consider filtering to only large states for production.

## Step 5 · Map Projections with alt.Projection

Vega-Lite supports all D3 projections. The `.project()` shorthand is equivalent to `.project(alt.Projection(type='...'))`. Common projections:

| Projection | Best for |
|---|---|
| `naturalEarth1` | World maps, area-balanced |
| `mercator` | Web maps, familiar but distorts poles |
| `albersUsa` | US-only, includes Alaska + Hawaii insets |
| `equalEarth` | World maps, equal-area |
| `orthographic` | Globe view from a point |
| `azimuthalEqualArea` | Polar regions or hemispheres |

In [ ]:
# Compare three world map projections side by side
projections = ["naturalEarth1", "mercator", "orthographic"]
proj_charts = []

for proj in projections:
    c = (
        alt.Chart(alt.topo_feature(world_topo_url, "countries"))
        .mark_geoshape(fill="#b5d0e8", stroke="white", strokeWidth=0.3)
        .project(
            alt.Projection(
                type=proj,
                rotate=[-10, 0, 0] if proj == "orthographic" else alt.Undefined,
            )
        )
        .properties(title=proj, width=220, height=180)
    )
    proj_charts.append(c)

# Display side by side
proj_charts[0] | proj_charts[1] | proj_charts[2]

**What just happened?**

- `alt.Projection(type='...', rotate=[lon, lat, roll])` gives full control over projection parameters.
- `rotate=[-10, 0, 0]` tilts the orthographic globe 10° east — centering it on the Atlantic.
- `alt.Undefined` is Altair's way of omitting a parameter — equivalent to not specifying it.
- **naturalEarth1 is the best general-purpose world projection** — it balances area and shape.

## Step 6 · Layer Point Marks (Cities) on the Choropleth

Overlay city or airport points on a choropleth by creating a second `alt.Chart` with `mark_point()` or `mark_circle()` and encoding `longitude`/`latitude`. Stack with `alt.layer()`.

In [ ]:
# Load airport data (has lat/lon for US airports)
airports = data.airports()
print("Airports columns:", airports.columns.tolist())
print(f"{len(airports)} airports")

# Filter to major airports (state capitals or large hubs)
# Use a simple filter: airports with iata codes of length 3
major_airports = airports.dropna(subset=["latitude", "longitude"]).head(50)

# Base: US state outlines (simpler — no choropleth data needed)
us_base = (
    alt.Chart(alt.topo_feature(us_states_url, "states"))
    .mark_geoshape(fill="#e8f0e8", stroke="#aaa", strokeWidth=0.5)
    .project("albersUsa")
)

# Airport point layer
airport_points = (
    alt.Chart(major_airports)
    .mark_circle(size=30, color="#E8890C", opacity=0.8)
    .encode(
        longitude="longitude:Q",
        latitude="latitude:Q",
        tooltip=["name:N", "iata:N", "state:N"],
    )
    .project("albersUsa")
)

# Compose
airport_map = (
    alt.layer(us_base, airport_points)
    .properties(
        title="US Airports — Major Hubs",
        width=650, height=380
    )
)

airport_map

**What just happened?**

- The point layer uses `longitude` and `latitude` as encoding channels — Altair applies the same projection.
- `alt.layer(base_map, points_layer)` is the standard pattern for overlaying marks on a map.
- **Both layers must share the same projection** — otherwise points appear at wrong locations.
- `tooltip` works on geoshape layers too — hover shows airport name, IATA code, and state.

In [ ]:
# Challenge: Build a world choropleth showing life expectancy
# Requirements:
#   1. Load the gapminder dataset and filter to year == 2000
#   2. Load the world_110m TopoJSON (data.world_110m.url)
#   3. The world_110m countries have a numeric 'id' property
#      Gapminder's country identifiers may need remapping.
#      HINT: use alt.topo_feature(url, 'countries') and
#            transform_lookup(lookup='id', from_=alt.LookupData(...))
#      HINT: you can also try using 'country' name as the join key if your
#            data source has it (the gapminder_all_years dataset in vega-datasets
#            may have a numeric id — inspect it first)
#   4. Color countries by 'life_expect' using the 'viridis' color scheme
#   5. Apply the 'naturalEarth1' projection
#   6. Add a tooltip showing country name and life_expect
#   7. Layer a second chart with mark_circle points for at least 5 cities
#      (create a small DataFrame with lat/lon/city name manually)
#   8. Set width=700, height=380

from vega_datasets import data as vd
import pandas as pd

# Inspect available geo datasets
gap = vd.gapminder()
gap_2000 = gap[gap['year'] == 2000]
print("Gapminder columns:", gap_2000.columns.tolist())
print(gap_2000.head(3))

# Major cities for point layer (fill these in)
cities = pd.DataFrame({
    'city': ['London', 'New York', 'Tokyo', 'Sydney', 'São Paulo'],
    'lat':  [51.5, 40.7, 35.7, -33.9, -23.5],
    'lon':  [-0.1, -74.0, 139.7, 151.2, -46.6],
})

# TODO: world_url = vd.world_110m.url
# TODO: countries_src = alt.topo_feature(world_url, 'countries')

# TODO: choropleth = (
#     alt.Chart(countries_src)
#     .mark_geoshape()
#     .encode(color=...)
#     .transform_lookup(lookup='id', from_=alt.LookupData(gap_2000, 'id', ['life_expect', 'country']))
#     .project('naturalEarth1')
#     .properties(width=700, height=380, title='Life Expectancy (2000)')
# )

# TODO: city_points = (
#     alt.Chart(cities)
#     .mark_circle(color='red', size=50)
#     .encode(longitude='lon:Q', latitude='lat:Q', tooltip=['city:N'])
#     .project('naturalEarth1')
# )

# TODO: alt.layer(choropleth, city_points)

print("Scaffold ready — fill in the TODOs above")

---
## Day 9 key concepts recap

| Concept | What to remember |
|---|---|
| `mark_geoshape()` | Renders TopoJSON/GeoJSON features as filled polygons |
| `alt.topo_feature(url, feature)` | Extracts a named feature collection from a TopoJSON |
| `transform_lookup` | Joins your DataFrame to GeoJSON features by a shared key |
| `longitude` / `latitude` encodings | Geographic channels — projection applied automatically |
| `alt.layer(base, points)` | Stack marks on top of a map; all layers must share the same projection |
| `.project('naturalEarth1')` | Applies a map projection; shorthand for `alt.Projection(type=...)` |
| `alt.Projection(rotate=[...])` | Fine-grained projection control — rotation, center, clip angle |

> **Tip:** Always use `transform_lookup` to join your data to the GeoJSON features — the TopoJSON topology by itself has no values, only geometry. Match on the feature id or name property.

---
## What's next
**Day 10** → Capstone — build a full multi-panel interactive EDA dashboard with linked brushing, cross-filtering, a choropleth panel, and a custom theme — all from the gapminder dataset.

Mark Day 9 complete in your [tracker](../index.html).